# 01_explore: Explore Allen Brain Atlas Data

This notebook explores the Allen Human Brain Atlas expression data
after processing with abagen.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from load_data import load_allen_expression

In [ ]:
# Load expression data
expression_matrix, donor_matrices = load_allen_expression(atlas="desikan-killiany")

print(f"Expression matrix shape: {expression_matrix.shape}")
print(f"Number of donors: {len(donor_matrices)}")
print(f"Regions: {expression_matrix.index.tolist()[:20]}")
print(f"Genes: {expression_matrix.columns.tolist()[:20]}")

In [ ]:
# Basic statistics
print("Expression matrix stats:")
print(expression_matrix.describe())

# Missing values
print(f"\nMissing values: {expression_matrix.isna().sum().sum()}")

# Per-donor shapes
for donor, mat in donor_matrices.items():
    print(f"{donor}: {mat.shape}")

In [ ]:
# Visualize expression distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Overall distribution
axes[0, 0].hist(expression_matrix.values.flatten(), bins=50, alpha=0.7)
axes[0, 0].set_title("Expression Value Distribution")
axes[0, 0].set_xlabel("Expression")
axes[0, 0].set_ylabel("Frequency")

# Per-region mean expression
region_means = expression_matrix.mean(axis=1).sort_values()
axes[0, 1].barh(range(len(region_means)), region_means.values)
axes[0, 1].set_yticks(range(len(region_means)))
axes[0, 1].set_yticklabels(region_means.index, fontsize=6)
axes[0, 1].set_title("Mean Expression per Region")
axes[0, 1].set_xlabel("Mean Expression")

# Per-gene mean expression
gene_means = expression_matrix.mean(axis=0).sort_values()
axes[1, 0].hist(gene_means.values, bins=50, alpha=0.7)
axes[1, 0].set_title("Mean Expression per Gene")
axes[1, 0].set_xlabel("Mean Expression")
axes[1, 0].set_ylabel("Frequency")

# Donor correlation
if len(donor_matrices) > 1:
    donor_names = list(donor_matrices.keys())
    corr_matrix = pd.DataFrame(index=donor_names, columns=donor_names)
    for d1 in donor_names:
        for d2 in donor_names:
            common_genes = set(donor_matrices[d1].columns) & set(donor_matrices[d2].columns)
            common_regions = set(donor_matrices[d1].index) & set(donor_matrices[d2].index)
            if len(common_genes) > 0 and len(common_regions) > 0:
                m1 = donor_matrices[d1].loc[list(common_regions), list(common_genes)].mean(axis=1)
                m2 = donor_matrices[d2].loc[list(common_regions), list(common_genes)].mean(axis=1)
                corr_matrix.loc[d1, d2] = m1.corr(m2)
    sns.heatmap(corr_matrix.astype(float), annot=True, cmap='RdBu_r', center=0, ax=axes[1, 1])
    axes[1, 1].set_title("Donor Correlation (Mean Regional Expression)")

plt.tight_layout()
plt.show()

## Notes

- Expression values are log2-intensities from microarray
- abagen handles probe-to-gene collapsing and donor normalization
- Desikan-Killiany atlas has 68 cortical + subcortical regions
- Substantia nigra is NOT in this atlas (known limitation)